# 01 — Pré-processamento e Panorama Geral

**Pergunta de pesquisa:** Qual é o perfil clínico de pacientes com e sem
diagnóstico de doença cardíaca, e quais indicadores se mostram mais
informativos para distinguir os dois grupos?

**Dataset:** Heart Failure Prediction Dataset — combinação de cinco bases
clínicas (Cleveland, Hungria, Suíça, Long Beach, Stalog).

---

## 1.1 Importações e configuração

In [ ]:
import sys
from pathlib import Path

# Garantir que src/ seja importável a partir do notebook
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import matplotlib.pyplot as plt

from src.config import (
    DATA_RAW, TARGET, TOTAL_N, QUAL_VARS, DISC_VAR, CONT_VARS,
    VAR_LABELS, CAT_LABELS, PALETTE_HD, PALETTE_HD_LIST,
)
from src.data_loader import load_raw, load_subsets

print(f"Projeto em: {PROJECT_ROOT}")
print(f"Dataset em: {DATA_RAW}")

## 1.2 Carga dos dados brutos

In [ ]:
df = load_raw()
print(f"Shape: {df.shape}")
print(f"Linhas: {df.shape[0]} | Colunas: {df.shape[1]}")
df.head(10)

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# Verificação: valores nulos explícitos
print("Valores nulos por coluna:")
print(df.isnull().sum())
print(f"\nTotal de nulos: {df.isnull().sum().sum()}")

## 1.3 Identificação de dados ausentes codificados como zero

Algumas variáveis contêm o valor zero como codificação de dado ausente,
não como valor real — zero é **biologicamente impossível** para essas
medidas clínicas.

In [ ]:
# Cholesterol == 0 → dado ausente (biologicamente impossível)
chol_zero = (df["Cholesterol"] == 0).sum()
print(f"Cholesterol = 0: {chol_zero} observações ({chol_zero/len(df)*100:.1f}%)")

# RestingBP == 0 → dado ausente (pressão arterial zero = impossível)
bp_zero = (df["RestingBP"] == 0).sum()
print(f"RestingBP = 0: {bp_zero} observação(ões) ({bp_zero/len(df)*100:.1f}%)")

# Mostrar a linha com RestingBP == 0
if bp_zero > 0:
    print("\nLinha(s) com RestingBP = 0:")
    display(df[df["RestingBP"] == 0])

### Decisões de pré-processamento

| Variável | Problema | Decisão | Justificativa |
|---|---|---|---|
| Cholesterol | 172 valores = 0 | Excluir dessas análises | Valor biologicamente impossível; imputar a média introduziria viés |
| RestingBP | 1 valor = 0 | Excluir dessa análise | Pressão arterial zero em paciente vivo é impossível |

As demais variáveis e observações permanecem intactas para suas respectivas
análises.

## 1.4 Criação dos subsets

In [ ]:
data = load_subsets()
df_full = data["df_full"]
df_chol = data["df_chol"]
df_bp   = data["df_bp"]

print(f"df_full: {df_full.shape[0]} linhas (dataset completo)")
print(f"df_chol: {df_chol.shape[0]} linhas (excluídos {TOTAL_N - df_chol.shape[0]} com Cholesterol=0)")
print(f"df_bp:   {df_bp.shape[0]} linhas (excluídos {TOTAL_N - df_bp.shape[0]} com RestingBP=0)")

# Validações
assert df_full.shape[0] == 918, f"Esperado 918, obtido {df_full.shape[0]}"
assert df_chol.shape[0] == 746, f"Esperado 746, obtido {df_chol.shape[0]}"
assert df_bp.shape[0] == 917, f"Esperado 917, obtido {df_bp.shape[0]}"
print("\n✓ Todas as validações passaram.")

## 1.5 Panorama geral — Quem são esses 918 pacientes?

In [ ]:
# Prevalência de doença cardíaca
hd_counts = df_full[TARGET].value_counts()
hd_pct = df_full[TARGET].value_counts(normalize=True) * 100

print("Distribuição de HeartDisease:")
for val in sorted(hd_counts.index):
    label = CAT_LABELS[TARGET][val]
    print(f"  {label}: {hd_counts[val]} ({hd_pct[val]:.1f}%)")

In [ ]:
# Gráfico de pizza — HeartDisease
fig, ax = plt.subplots(figsize=(6, 6))
labels_hd = [CAT_LABELS[TARGET][v] for v in sorted(hd_counts.index)]
values_hd = [hd_counts[v] for v in sorted(hd_counts.index)]
colors_hd = PALETTE_HD_LIST

wedges, texts, autotexts = ax.pie(
    values_hd, labels=labels_hd, colors=colors_hd,
    autopct=lambda p: f"{p:.1f}%\n({int(round(p * sum(values_hd) / 100))})",
    startangle=90, pctdistance=0.55,
    wedgeprops=dict(edgecolor="white", linewidth=2),
    textprops=dict(fontsize=12),
)
for at in autotexts:
    at.set_fontweight("bold")
ax.set_title("Prevalência de Doença Cardíaca na Amostra", fontsize=14, fontweight="bold")

fig.tight_layout()
fig.savefig(PROJECT_ROOT / "output" / "figures" / "00_prevalencia_hd.png",
            dpi=300, bbox_inches="tight", facecolor="white")
plt.show()

In [ ]:
# Distribuição por sexo
sex_counts = df_full["Sex"].value_counts()
sex_pct = df_full["Sex"].value_counts(normalize=True) * 100

print("Distribuição por sexo:")
for val in sex_counts.index:
    label = CAT_LABELS["Sex"][val]
    print(f"  {label}: {sex_counts[val]} ({sex_pct[val]:.1f}%)")

# Resumo de Age
print(f"\nIdade — mín: {df_full['Age'].min()}, máx: {df_full['Age'].max()}, "
      f"média: {df_full['Age'].mean():.1f}, mediana: {df_full['Age'].median():.0f}")

## 1.6 Variáveis do dataset

Resumo das 12 variáveis e seus tipos para orientar os próximos notebooks.

In [ ]:
var_info = pd.DataFrame({
    "Variável": df.columns,
    "Tipo (pandas)": [str(df[c].dtype) for c in df.columns],
    "Rótulo": [VAR_LABELS.get(c, c) for c in df.columns],
    "Bloco de análise": [
        "Qualitativa" if c in QUAL_VARS else
        "Quant. discreta" if c == DISC_VAR else
        "Quant. contínua" if c in CONT_VARS else
        "Variável-alvo"
        for c in df.columns
    ],
    "Valores únicos": [df[c].nunique() for c in df.columns],
})
var_info

---

**Próximo notebook:** `02_qualitative.ipynb` — Análise das variáveis qualitativas.